# seleccion_modelo

Proyecto ARIMA / SARIMA / ARIMAX / SARIMAX
Modelación epidemiológica con variables meteorológicas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import PowerTransformer, StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

In [3]:
# ============================================================
# 1. CARGA DE BASE DE DATOS ORIGINAL
# ============================================================
ruta_excel = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\datos_consolidados\1_datos_semana_epi_meteo\datos_semanal_meteo_epi.xlsx"
df_orig = pd.read_excel(ruta_excel)
df_orig['fecha'] = pd.to_datetime(df_orig['fecha'])
df_orig.set_index('fecha', inplace=True)

df_transf = df_orig[['año', 'semana_epi']].copy()

In [4]:
# ============================================================
# 2. APLICACIÓN DE LAS TRANSFORMACIONES SOLICITADAS
# ============================================================
# A. Logaritmo Natural (log1p para manejar ceros de forma segura)
df_transf['casos_ln'] = np.log1p(df_orig['casos_dengue'])
df_transf['prec_ln'] = np.log1p(df_orig['prec'])
df_transf['dias_lluvia_ln'] = np.log1p(df_orig['dias_lluvia'])

In [5]:
# B. Box-Cox para variables meteorológicas locales
pt_boxcox = PowerTransformer(method='box-cox')
vars_local = ['temp_max', 'hum_esp', 'hum_rel', 'vel_vi', 'vel_vi_max']
# Asegurar que no tengan valores <= 0 para Box-Cox (sumar un pequeño valor si es necesario)
for var in vars_local:
    datos_var = df_orig[[var]].copy()
    if (datos_var <= 0).any().any():
        datos_var = datos_var - datos_var.min() + 0.01
    df_transf[f'{var}_bc'] = pt_boxcox.fit_transform(datos_var)

In [6]:
# C. Yeo-Johnson para variables macroclimáticas (permite valores negativos de origen)
pt_yeo = PowerTransformer(method='yeo-johnson')
vars_macro = ['soi', 'sst']
for var in vars_macro:
    df_transf[f'{var}_yj'] = pt_yeo.fit_transform(df_orig[[var]])

In [7]:
# ============================================================
# 3. CREACIÓN DE LOS REZAGOS TEMPORALES (MAX LAG = 20)
# ============================================================
variables_base_lags = ['prec_ln', 'dias_lluvia_ln', 'temp_max_bc', 'hum_esp_bc', 'hum_rel_bc', 'vel_vi_bc', 'vel_vi_max_bc', 'sst_yj']
max_lag = 20

lags_dict = {}
for lag in range(1, max_lag + 1):
    for var in variables_base_lags:
        lags_dict[f'{var}_lag_{lag}'] = df_transf[var].shift(lag)

df_lags = pd.DataFrame(lags_dict, index=df_transf.index)
df_total = pd.concat([df_transf[['año', 'semana_epi', 'casos_ln']], df_lags], axis=1).dropna()

In [8]:
# Tus 30 rezagos epidemiológicos validados por la literatura
variables_importantes = [
    'hum_esp_bc_lag_2','hum_esp_bc_lag_3','hum_esp_bc_lag_4','hum_esp_bc_lag_5','hum_esp_bc_lag_6','hum_esp_bc_lag_7','hum_esp_bc_lag_8',
    'prec_ln_lag_4','prec_ln_lag_5','prec_ln_lag_6','prec_ln_lag_7','prec_ln_lag_8',
    'dias_lluvia_ln_lag_4','dias_lluvia_ln_lag_5','dias_lluvia_ln_lag_6',
    'vel_vi_max_bc_lag_2','vel_vi_max_bc_lag_3','vel_vi_max_bc_lag_4','vel_vi_max_bc_lag_5','vel_vi_max_bc_lag_6',
    'temp_max_bc_lag_5','temp_max_bc_lag_6','temp_max_bc_lag_7','temp_max_bc_lag_8',
    'sst_yj_lag_16','sst_yj_lag_17','sst_yj_lag_18','sst_yj_lag_19','sst_yj_lag_20'
]

X_climaticas = df_total[variables_importantes]
y_transf = df_total['casos_ln']
metadatos_tiempo = df_total[['año', 'semana_epi']]

In [9]:
# Guardamos el vector original alineado de casos para la reversión exacta de las métricas
y_original_sivigila = df_orig.loc[df_total.index, 'casos_dengue']

In [10]:
# ============================================================
# 4. PARÁMETROS DE LOS MODELOS (TUS ÓRDENES)
# ============================================================
arima_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1), (1,1,2), (2,1,2)]
sarima_orders = [((1,1,1),(1,1,1,52)), ((1,1,0),(1,1,1,52)), ((0,1,1),(1,1,1,52))]
arimax_orders = [(1,1,0), (0,1,1), (1,1,1), (2,1,1), (1,1,2)]
sarimax_orders = [((1,1,1),(1,1,1,52)), ((0,1,1),(1,1,1,52))]

porcentajes_train = [0.80, 0.90, 0.95]
todas_las_metricas = []
mejores_predicciones = {}

In [11]:
# ============================================================
# 5. BUCLE DE EXPERIMENTACIÓN MULTI-ESCALA
# ============================================================
for p_train in porcentajes_train:
    split_idx = int(len(df_total) * p_train)
    
    # Divisiones temporales estrictas
    X_train_c, X_test_c = X_climaticas.iloc[:split_idx], X_climaticas.iloc[split_idx:]
    y_train, y_test = y_transf.iloc[:split_idx], y_transf.iloc[split_idx:]
    time_train, time_test = metadatos_tiempo.iloc[:split_idx], metadatos_tiempo.iloc[split_idx:]
    y_train_real, y_test_real = y_original_sivigila.iloc[:split_idx], y_original_sivigila.iloc[split_idx:]
    
    # ESTANDARIZACIÓN LOCAL DE VARIABLES EXÓGENAS (Evita Data Leakage)
    scaler_X = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train_c), columns=variables_importantes, index=y_train.index)
    X_test_scaled = pd.DataFrame(scaler_X.transform(X_test_c), columns=variables_importantes, index=y_test.index)
    
    # ESTANDARIZACIÓN LOCAL DE LA VARIABLE OBJETIVO (Requerida para un LASSO estable)
    scaler_y = StandardScaler()
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()
    
    # Convertir a Series para conservar índices temporales
    y_train_scaled_series = pd.Series(y_train_scaled, index=y_train.index)
    y_test_scaled_series = pd.Series(y_test_scaled, index=y_test.index)
    
    # Ajuste de LASSO CV para selección de características
    lasso_model = LassoCV(cv=5, random_state=42).fit(X_train_scaled, y_train_scaled_series)
    
    # Predicciones de LASSO Puro (Escala Estandarizada)
    pred_lasso_tr_scaled = lasso_model.predict(X_train_scaled)
    pred_lasso_te_scaled = lasso_model.predict(X_test_scaled)
    
    # Función inversa local para regresar de la estandarización y del logaritmo
    def inversa_total_local(y_scaled_val):
        # 1. Deshacer Z-score
        y_ln_val = scaler_y.inverse_transform(np.array(y_scaled_val).reshape(-1, 1)).flatten()
        # 2. Deshacer Logaritmo Natural (expm1 revierte log1p de forma exacta)
        return np.expm1(y_ln_val)
    
    pred_lasso_tr_real = inversa_total_local(pred_lasso_tr_scaled)
    pred_lasso_te_real = inversa_total_local(pred_lasso_te_scaled)
    
    # Guardar métricas del Quinto Modelo (LASSO Puro)
    todas_las_metricas.append({
        'Split_Train': f'{int(p_train*100)}%', 'Modelo': 'LASSO_Puro', 'Configuración': f'alpha={lasso_model.alpha_:.4f}',
        'MAE_Train': mean_absolute_error(y_train_real, pred_lasso_tr_real), 'MAE_Test': mean_absolute_error(y_test_real, pred_lasso_te_real),
        'RMSE_Train': np.sqrt(mean_squared_error(y_train_real, pred_lasso_tr_real)), 'RMSE_Test': np.sqrt(mean_squared_error(y_test_real, pred_lasso_te_real)),
        'R2_Train': r2_score(y_train_scaled_series, pred_lasso_tr_scaled), 'R2_Test': r2_score(y_test_scaled_series, pred_lasso_te_scaled)
    })
    
    # Filtrado de variables exógenas mediante coeficientes de LASSO
    coeficientes = pd.Series(lasso_model.coef_, index=variables_importantes)
    vars_ganadoras = coeficientes[coeficientes != 0].index.tolist()
    
    if len(vars_ganadoras) == 0:
        vars_ganadoras = coeficientes.abs().nlargest(3).index.tolist()
        
    X_train_lasso_sub = X_train_scaled[vars_ganadoras]
    X_test_lasso_sub = X_test_scaled[vars_ganadoras]
    
    # Configuración de los experimentos de la familia ARIMA (Trabajan sobre y_train en escala Logarítmica)
    experimentos = [
        {'tipo': 'ARIMA', 'ordenes': [(o, (0,0,0,0)) for o in arima_orders], 'exog_tr': None, 'exog_te': None},
        {'tipo': 'SARIMA', 'ordenes': sarima_orders, 'exog_tr': None, 'exog_te': None},
        {'tipo': 'ARIMAX', 'ordenes': [(o, (0,0,0,0)) for o in arimax_orders], 'exog_tr': X_train_lasso_sub, 'exog_te': X_test_lasso_sub},
        {'tipo': 'SARIMAX', 'ordenes': sarimax_orders, 'exog_tr': X_train_lasso_sub, 'exog_te': X_test_lasso_sub}
    ]
    
    for exp in experimentos:
        for orden_p, orden_s in exp['ordenes']:
            try:
                # Los modelos ARIMA estiman mejor directamente sobre la escala y_train (log)
                model = SARIMAX(endog=y_train, exog=exp['exog_tr'], order=orden_p, seasonal_order=orden_s)
                fitted_model = model.fit(disp=False)
                
                pred_tr_ln = pd.Series(fitted_model.fittedvalues, index=y_train.index)
                pred_te_ln = pd.Series(fitted_model.predict(start=y_test.index[0], end=y_test.index[-1], exog=exp['exog_te']), index=y_test.index)
                
                # Reversión de la escala logarítmica para obtener personas reales
                pred_tr_real = np.expm1(pred_tr_ln)
                pred_te_real = np.expm1(pred_te_ln)
                
                # Cálculo de R2 en la escala interna de entrenamiento para mantener consistencia estadística
                r2_tr = r2_score(y_train.iloc[1:], pred_tr_ln.iloc[1:])
                r2_te = r2_score(y_test, pred_te_ln)
                
                # Métricas MAE y RMSE calculadas estrictamente sobre número real de casos de Dengue
                mae_tr = mean_absolute_error(y_train_real.iloc[1:], pred_tr_real.iloc[1:])
                mae_te = mean_absolute_error(y_test_real, pred_te_real)
                rmse_tr = np.sqrt(mean_squared_error(y_train_real.iloc[1:], pred_tr_real.iloc[1:]))
                rmse_te = np.sqrt(mean_squared_error(y_test_real, pred_te_real))
                
                str_orden = f"order={orden_p}" if exp['tipo'] in ['ARIMA','ARIMAX'] else f"order={orden_p} seasonal={orden_s}"
                
                todas_las_metricas.append({
                    'Split_Train': f'{int(p_train*100)}%', 'Modelo': exp['tipo'], 'Configuración': str_orden,
                    'MAE_Train': mae_tr, 'MAE_Test': mae_te, 'RMSE_Train': rmse_tr, 'RMSE_Test': rmse_te,
                    'R2_Train': r2_tr, 'R2_Test': r2_te
                })
                
                df_pred = time_test.copy()
                df_pred['Real'] = y_test_real
                df_pred['Predicho'] = pred_te_real
                mejores_predicciones[f"{exp['tipo']}_{int(p_train*100)}_{str_orden}"] = df_pred
            except:
                continue

In [12]:
# ============================================================
# 6. GENERACIÓN DE LA TABLA COMPARATIVA FINAL DE TESIS
# ============================================================
df_completo = pd.DataFrame(todas_las_metricas)
mejores_modelos_resumen = df_completo.loc[df_completo.groupby(['Split_Train', 'Modelo'])['MAE_Test'].idxmin()]

print("\n" + "="*95 + "\n   TABLA DE RESULTADOS DE TESIS ACTUALIZADA (DATOS ORIGINALES PROCESADOS CORRECTAMENTE) \n" + "="*95)
print(mejores_modelos_resumen.to_string(index=False))


   TABLA DE RESULTADOS DE TESIS ACTUALIZADA (DATOS ORIGINALES PROCESADOS CORRECTAMENTE) 
Split_Train     Modelo                          Configuración  MAE_Train  MAE_Test  RMSE_Train  RMSE_Test  R2_Train   R2_Test
        80%      ARIMA                        order=(2, 1, 1)   5.241542 41.454647    8.017293  46.757394  0.835251 -1.472988
        80%     ARIMAX                        order=(1, 1, 2)   5.424040 33.073336    8.540444  38.757262  0.853967 -0.985356
        80% LASSO_Puro                           alpha=0.0177   9.923071 29.365473   16.793823  38.777147  0.644140 -2.330644
        80%     SARIMA order=(1, 1, 1) seasonal=(1, 1, 1, 52)   7.633059 75.504830   11.205723  94.824022  0.687285 -3.597342
        80%    SARIMAX order=(1, 1, 1) seasonal=(1, 1, 1, 52)   7.631242 54.261494   11.351872  68.646533  0.721569 -2.497097
        90%      ARIMA                        order=(1, 1, 0)   5.785419 17.550781    8.686323  20.385882  0.854597 -1.282698
        90%     ARIMAX      